In [1]:
# =====================================================
# SanketSetu Transformer V5.2 - Imports & Dataset
# =====================================================

import os
import random
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("TensorFlow Version:", tf.__version__)

# ---------------- Dataset ----------------

DATA_PATH = "../dataset/training"

X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))

print("✅ Balanced Dataset Loaded!")
print("Train :", X_train.shape, y_train.shape)
print("Val   :", X_val.shape, y_val.shape)
print("Test  :", X_test.shape, y_test.shape)

num_classes = len(np.unique(y_train))
sequence_length = X_train.shape[1]
feature_dim = X_train.shape[2]

print("Classes :", num_classes)
print("Sequence Length :", sequence_length)
print("Features :", feature_dim)

TensorFlow Version: 2.21.0
✅ Balanced Dataset Loaded!
Train : (7320, 154, 63) (7320,)
Val   : (630, 154, 63) (630,)
Test  : (420, 154, 63) (420,)
Classes : 210
Sequence Length : 154
Features : 63


In [5]:
# ===============================================
# Transformer V5.1 - Gentle Augmentation Function
# ===============================================

def augment_sequence(sequence):
    seq = sequence.copy()

    # Pick ONE augmentation randomly
    choice = np.random.choice(["rotate", "scale", "translate", "noise"])

    if choice == "rotate":
        angle = np.deg2rad(np.random.uniform(-5, 5))
        c, s = np.cos(angle), np.sin(angle)

        for i in range(21):
            x = seq[:, i * 3]
            y = seq[:, i * 3 + 1]

            seq[:, i * 3] = x * c - y * s
            seq[:, i * 3 + 1] = x * s + y * c

    elif choice == "scale":
        scale = np.random.uniform(0.98, 1.02)
        seq *= scale

    elif choice == "translate":
        tx = np.random.uniform(-0.01, 0.01)
        ty = np.random.uniform(-0.01, 0.01)

        for i in range(21):
            seq[:, i * 3] += tx
            seq[:, i * 3 + 1] += ty

    elif choice == "noise":
        seq += np.random.normal(0, 0.0015, seq.shape)

    return seq

print("✅ Gentle augmentation function ready!")

✅ Gentle augmentation function ready!


In [10]:
# ============================================
# Reload ORIGINAL training dataset
# ============================================

DATA_PATH = "../dataset/training"

X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

print("✅ Original dataset restored!")
print("Train shape:", X_train.shape)
print("Labels shape:", y_train.shape)

✅ Original dataset restored!
Train shape: (7320, 154, 63)
Labels shape: (7320,)


In [20]:
import os
import numpy as np

DATA_PATH = "../dataset/training"

# Reload original datasets from disk
X_train = np.load(os.path.join(DATA_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATA_PATH, "y_train.npy"))

X_val = np.load(os.path.join(DATA_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATA_PATH, "y_val.npy"))

X_test = np.load(os.path.join(DATA_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATA_PATH, "y_test.npy"))

print("Fresh datasets loaded.")
print(X_train.shape, X_val.shape, X_test.shape)

Fresh datasets loaded.
(7320, 154, 63) (630, 154, 63) (420, 154, 63)


In [21]:
# ============================================
# Transformer V5.1 — Controlled Augmentation
# ============================================

augmented_X = []
augmented_y = []

augmentation_ratio = 0.35   # Augment only 35% of training samples

for seq, label in zip(X_train, y_train):

    # Keep original sample
    augmented_X.append(seq)
    augmented_y.append(label)

    # Augment only some samples
    if np.random.rand() < augmentation_ratio:
        augmented_X.append(augment_sequence(seq))
        augmented_y.append(label)

# Convert to NumPy arrays
X_train = np.array(augmented_X, dtype=np.float32)
y_train = np.array(augmented_y)

# Shuffle dataset
perm = np.random.permutation(len(X_train))
X_train = X_train[perm]
y_train = y_train[perm]

print("✅ Controlled augmentation complete!")
print("Augmentation Ratio :", augmentation_ratio)
print("New Train Shape     :", X_train.shape)
print("Labels Shape        :", y_train.shape)

✅ Controlled augmentation complete!
Augmentation Ratio : 0.35
New Train Shape     : (9914, 154, 63)
Labels Shape        : (9914,)


In [22]:
# Compute statistics ONLY from augmented training set
train_mean = X_train.mean(axis=(0,1), keepdims=True)
train_std = X_train.std(axis=(0,1), keepdims=True) + 1e-8

# Normalize once
X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

print("Normalization complete.")

print("Train std:", round(float(X_train.std()),4))
print("Val std  :", round(float(X_val.std()),4))
print("Test std :", round(float(X_test.std()),4))

Normalization complete.
Train std: 0.9997
Val std  : 1.0035
Test std : 0.9921


In [23]:
print("Train std:", X_train.std())
print("Val std:", X_val.std())
print("Test std:", X_test.std())

Train std: 0.9997399
Val std: 1.0034806
Test std: 0.99207723


In [24]:
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Steps per epoch:", len(X_train)//32)

Training samples: 9914
Validation samples: 630
Steps per epoch: 309


In [25]:
# ==========================================
# Transformer V5.1 - Compile + Train
# ==========================================

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-4,
    decay_steps=60 * (len(X_train) // 32),
    alpha=0.1
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=12,
    restore_best_weights=True,
    mode="max",
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "../models/best_transformer_v5_2.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

print("✅ Transformer V5.1 training complete!")

Epoch 1/60
310/310 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step - accuracy: 0.7018 - loss: 0.9776
Epoch 1: val_accuracy improved from None to 0.70000, saving model to ../models/best_transformer_v5_2.keras

Epoch 1: finished saving model to ../models/best_transformer_v5_2.keras
310/310 ━━━━━━━━━━━━━━━━━━━━ 147s 451ms/step - accuracy: 0.7018 - loss: 0.9776 - val_accuracy: 0.7000 - val_loss: 1.0333
Epoch 2/60
310/310 ━━━━━━━━━━━━━━━━━━━━ 0s 536ms/step - accuracy: 0.7335 - loss: 0.8494
Epoch 2: val_accuracy did not improve from 0.70000
310/310 ━━━━━━━━━━━━━━━━━━━━ 169s 547ms/step - accuracy: 0.7335 - loss: 0.8494 - val_accuracy: 0.6889 - val_loss: 1.0283
Epoch 3/60
310/310 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step - accuracy: 0.7526 - loss: 0.7884
Epoch 3: val_accuracy improved from 0.70000 to 0.75873, saving model to ../models/best_transformer_v5_2.keras

Epoch 3: finished saving model to ../models/best_transformer_v5_2.keras
310/310 ━━━━━━━━━━━━━━━━━━━━ 137s 441ms/step - accuracy: 0.7526 - loss: 0.7884 -

In [1]:
print("train_mean" in globals())
print("train_std" in globals())

False
False


In [2]:
import numpy as np
import tensorflow as tf

# Load the saved datasets
X_train_saved = np.load("../dataset/training/X_train.npy")
X_test_saved = np.load("../dataset/training/X_test.npy")
y_test_saved = np.load("../dataset/training/y_test.npy")

# Calculate statistics from the saved training set
mean_saved = X_train_saved.mean(axis=(0, 1), keepdims=True)
std_saved = X_train_saved.std(axis=(0, 1), keepdims=True) + 1e-8

# Normalize test set
X_test_norm = (X_test_saved - mean_saved) / std_saved

# Load FINAL V5.2 model
model = tf.keras.models.load_model(
    "../models/best_transformer_v5_2.keras"
)

# Evaluate
loss, accuracy = model.evaluate(
    X_test_norm,
    y_test_saved,
    verbose=1
)

print("Test accuracy using saved X_train statistics:", accuracy)

14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 140ms/step - accuracy: 0.8976 - loss: 0.4568
Test accuracy using saved X_train statistics: 0.8976190686225891


In [3]:
np.save("../outputs/v5_2_train_mean.npy", mean_saved)
np.save("../outputs/v5_2_train_std.npy", std_saved)

print("V5.2 normalization statistics saved.")
print("Mean shape:", mean_saved.shape)
print("Std shape:", std_saved.shape)

V5.2 normalization statistics saved.
Mean shape: (1, 1, 63)
Std shape: (1, 1, 63)


In [1]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "../models/best_transformer_v5_2.keras"
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 154, 63)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 154, 128)  │      8,192 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ dense[0][0],      │
│ (MultiHeadAttentio… │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 154, 128)  │          0 │ dense[0][0],      │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 154, 128)  │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 154, 256)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 154, 128)  │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 154, 128)  │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 154, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 154, 256)  │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 154, 128)  │     32,896 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 154, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 154, 128)  │        256 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                 

 Total params: 950,263 (3.62 MB)

 Trainable params: 316,754 (1.21 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 633,509 (2.42 MB)

In [7]:
import numpy as np
import json
import tensorflow as tf

# ============================================================
# LOAD ACTUAL LIVE GOOD SEQUENCE
# ============================================================

MODEL_PATH = "../models/best_transformer_v5_2.keras"
MEAN_PATH = "../outputs/v5_2_train_mean.npy"
STD_PATH = "../outputs/v5_2_train_std.npy"
MAPPING_PATH = "../outputs/class_mapping.json"
SOURCE = "../outputs/live_sequence.npy"

model = tf.keras.models.load_model(MODEL_PATH)

mean = np.squeeze(np.load(MEAN_PATH))
std = np.squeeze(np.load(STD_PATH))

with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

sequence = np.load(SOURCE).astype(np.float32)

print("Sequence:", sequence.shape)
print("Mean:", mean.shape)
print("Std:", std.shape)

assert sequence.shape == (154, 63)


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_test(seq, name):

    x = (seq - mean) / std

    probs = model.predict(
        x[np.newaxis, ...],
        verbose=0
    )[0]

    top5 = np.argsort(probs)[-5:][::-1]

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    for rank, idx in enumerate(top5, 1):
        gesture = class_mapping.get(str(int(idx)), "UNKNOWN")

        print(
            f"{rank}. Class {idx:3d} | "
            f"{gesture:25s} | "
            f"{probs[idx] * 100:.3f}%"
        )

    best = int(np.argmax(probs))

    print(
        f"\nPREDICTION: "
        f"{class_mapping.get(str(best), 'UNKNOWN')}"
    )
    print(f"CONFIDENCE: {probs[best] * 100:.3f}%")


# ============================================================
# TEST 1 — ORIGINAL LIVE SEQUENCE
# ============================================================

predict_test(
    sequence,
    "TEST 1 — ORIGINAL LIVE (154 frames)"
)


# ============================================================
# FIND MOST ACTIVE WINDOWS
# ============================================================

movement = np.linalg.norm(
    sequence[1:] - sequence[:-1],
    axis=1
)

print("\nMovement statistics:")
print("50% :", np.percentile(movement, 50))
print("75% :", np.percentile(movement, 75))
print("90% :", np.percentile(movement, 90))
print("95% :", np.percentile(movement, 95))
print("99% :", np.percentile(movement, 99))


def find_active_window(window_size):

    scores = np.convolve(
        movement,
        np.ones(window_size - 1),
        mode="valid"
    )

    start = int(np.argmax(scores))
    end = start + window_size

    return start, end, scores[start]


# ============================================================
# TEST 2–4 — ACTIVE GESTURE + ZERO PADDING
# ============================================================

for n in [60, 80, 100]:

    start, end, score = find_active_window(n)

    test_seq = np.zeros(
        (154, 63),
        dtype=np.float32
    )

    test_seq[:n] = sequence[start:end]

    print(
        f"\nActive window: frames {start} → {end}"
        f" | movement score = {score:.3f}"
    )

    predict_test(
        test_seq,
        f"TEST — {n} ACTIVE FRAMES + {154-n} ZERO FRAMES"
    )

Sequence: (154, 63)
Mean: (63,)
Std: (63,)

TEST 1 — ORIGINAL LIVE (154 frames)
1. Class   0 | 1. Dog                    | 68.189%
2. Class 183 | 87. hot                   | 18.030%
3. Class   1 | 1. loud                   | 5.864%
4. Class  97 | 53. Good evening          | 2.803%
5. Class 208 | 99. healthy               | 1.723%

PREDICTION: 1. Dog
CONFIDENCE: 68.189%

Movement statistics:
50% : 0.021095260977745056
75% : 0.07139425724744797
90% : 0.13400563150644318
95% : 0.25079551339149553
99% : 5.428824596405029

Active window: frames 1 → 61 | movement score = 35.763

TEST — 60 ACTIVE FRAMES + 94 ZERO FRAMES
1. Class 208 | 99. healthy               | 99.671%
2. Class 175 | 84. small little          | 0.275%
3. Class  97 | 53. Good evening          | 0.023%
4. Class 183 | 87. hot                   | 0.010%
5. Class 178 | 85. slow                  | 0.008%

PREDICTION: 99. healthy
CONFIDENCE: 99.671%

Active window: frames 0 → 80 | movement score = 36.541

TEST — 80 ACTIVE FRAMES + 